# ハンズオン(1)可視化・診断編 — 非線形物理モデルの「なぜ遅いか」を読む

このノートは `minlpkit` の **診断エンジンが何を見ているか** を、実サンプルで手を動かして理解するためのもの。

題材は **地域熱供給網の詳細物理モデル**
(`samples/physics_and_control_minlp/district_heating_detailed_physics.py`)。
配管内の質量流量 × 温度(双線形)、圧力損失(流量の2乗)、ノードでの温度混合など、
**連続変数どうしの積・2乗**を多数含む MINLP である。

### なぜこのモデルなのか(診断センサスの結果)

4カテゴリ約50本のサンプルに `mk.analyze` を一括適用した
[診断センサス](../census.md)では、最も重い症状
`weak_relaxation`(緩和が弱く空間分枝に頼る)が発火したのは**このモデル1本だけ**だった。
現代のSCIPは大半のモデルを賢く解いてしまうので、
「非凸緩和の弱さが本当に律速になっている」ケースは貴重で、診断の題材として最適である。

流れ:
1. モデルを読み込む
2. `mk.analyze` で観測+診断 → `findings` / `recipe` の読み方
3. 診断が内部で見ている**生の観測量**(違反ヒートマップ・空間分枝の寄与)を直接叩く
4. 「診断が何を根拠に weak_relaxation と言っているか」を突き合わせる

In [1]:
import sys, pathlib
# リポジトリルート(samples/ を持つ階層)を探して import パスに載せる。
# nbconvert 実行時の cwd は notebook のディレクトリなので、上に辿って解決する。
ROOT = pathlib.Path.cwd()
while not (ROOT / "samples").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ["samples", "samples/physics_and_control_minlp", "samples/others"]:
    p = str(ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)
print("repo root:", ROOT)

repo root: C:\work_space\mip


## 1. モデルを読み込む

`build_model()` を import するだけ。この題材は引数なしで動く。
どんな非線形制約を持つかを覗いておく。

In [2]:
import minlpkit as mk
import district_heating_detailed_physics as dh

m = dh.build_model()
n_nl = sum(1 for c in m.getConss() if c.isNonlinear())
print(f"変数 {m.getNVars()} 本 / 制約 {m.getNConss()} 本 / うち非線形 {n_nl} 本")
print("非線形制約の例:", [c.name for c in m.getConss() if c.isNonlinear()][:6])

変数 13 本 / 制約 12 本 / うち非線形 10 本
非線形制約の例: ['demand_heat_1', 'demand_heat_2', 'heat_loss_0_1', 'heat_loss_1_2', 'temp_mix_1', 'temp_mix_2']


## 2. `mk.analyze` — 観測して診断する

`analyze` はモデルを実際に解き、双対境界の推移・空間分枝の偏り・非線形制約の違反・
係数スケールなどを収集して、診断ルールに照らした **症状(findings)** を重要度順に返す。

In [3]:
report = mk.analyze(dh.build_model, name="district_heating", time_limit=10)
print(report.summary())

[district_heating] 検出症状 2件:
  [serious] 特定の非線形制約に緩和違反が集中(かつ空間分枝が多い) -> ボトルネック制約の区分線形近似・凸包再定式化・変数境界タイト化
  [warning] 係数の絶対値レンジが桁違い / Big-M候補あり(presolve後も残存) -> スケーリング、Big-MのIndicator/SOS制約化、係数の再定式化


### findings と recipe の読み方

各 finding は「症状 → 原因 → 推薦 → **具体的な直し方(recipe)** → 根拠(evidence)」を持つ。
recipe は「どの `mk` 関数でどう直すか」まで踏み込んだ実行可能な手順である。

In [4]:
for f in report.findings:
    print(f"■ [{f['severity']}] {f['id']}")
    print(f"  症状 : {f['symptom']}")
    print(f"  原因 : {f['cause']}")
    print(f"  推薦 : {f['recommendation']}")
    print(f"  根拠 : {f['evidence']}")
    print(f"  手順 : {f['recipe']}")
    print()

■ [serious] weak_relaxation
  症状 : 特定の非線形制約に緩和違反が集中(かつ空間分枝が多い)
  原因 : その制約の凸緩和が支配的ボトルネック(非凸項の緩和が緩い)
  推薦 : ボトルネック制約の区分線形近似・凸包再定式化・変数境界タイト化
  根拠 : 支配ボトルネック=demand_heat(相対違反26.00)、空間分枝の双対寄与97%
  手順 : 整数×連続の積は mk.linearize_product(m,y,x,...) で厳密線形化。非線形1変数は mk.pwl_sos2(m,x,brks,vals) で区分線形化。例: scheduling_plant(n·s)→improve_linearize.html, sos.html

■ [warning] numerical_scale
  症状 : 係数の絶対値レンジが桁違い / Big-M候補あり(presolve後も残存)
  原因 : 数値的不安定(丸め誤差でソルバーが迷走)。SCIPのpresolveで締まらない残存分
  推薦 : スケーリング、Big-MのIndicator/SOS制約化、係数の再定式化
  根拠 : presolve後の係数比=1.85e+06(presolve前120)、残存Big-M3件
  手順 : Big-Mを実bound/Indicator/SOSに置換。半連続on-offは Indicator、区分線形は mk.pwl_sos2。条件数は viz.static_diag.matrix_condition/scip_basis_condition で確認。例: sos.html, condition.html



`weak_relaxation`(重大)と `numerical_scale`(注意)が発火する。
前者は「非線形制約の緩和違反が特定の制約に集中し、かつ空間分枝が多い」ときに出る症状で、
このモデルの本質的な難しさ=**双線形項の McCormick 緩和が弱い**ことを指している。

## 3. 診断が内部で見ている観測量を直接叩く

診断ルールはブラックボックスではない。`weak_relaxation` の発火条件は
「支配ボトルネックの相対違反 `bottleneck_rel_viol ≥ 0.5`
**かつ** 空間分枝の寄与 `spatial_share ≥ 0.3`」である。
その2つの生の観測量を自分で計算して、診断の根拠を確かめる。

まず `report.metrics` に集約済みの値を見る。

In [5]:
keys = ["gap", "nodes", "nsols", "spatial_share",
        "bottleneck_type", "bottleneck_rel_viol",
        "residual_coef_ratio", "residual_bigm_count", "has_nonlinear"]
for k in keys:
    print(f"{k:24s} = {report.metrics.get(k)}")

gap                      = 3.8290873328821045e-05
nodes                    = 73
nsols                    = 16
spatial_share            = 0.9745112222048518
bottleneck_type          = demand_heat
bottleneck_rel_viol      = 25.99577986587965
residual_coef_ratio      = 1848314.602879209
residual_bigm_count      = 3
has_nonlinear            = True


### 3a. どの非線形制約が緩和のボトルネックか(違反ヒートマップの素データ)

`collect_root_violations` はルートLP緩和解を作り、各非線形制約の**違反量**を
`getNlRowSolFeasibility` で測る。相対違反 = 違反量 /(|活動値|+1)。
`violation_by_type` は制約タイプ別に集計する。これが診断ダッシュボードの
「違反ヒートマップ」の中身そのものである。

In [6]:
from minlpkit.collectors.violation import collect_root_violations, violation_by_type

vdf = collect_root_violations(dh.build_model())
by_type = violation_by_type(vdf)
print(by_type.to_string(index=False))

          ctype  mean_rel   max_rel  n
    demand_heat 25.995780 51.493232  2
    heat_loss_1  1.000000  1.000000  1
         source  0.944478  0.944478  1
       temp_mix  0.903412  0.993047  2
pressure_loss_0  0.768301  0.768301  1
pressure_loss_1  0.715234  0.715234  1
    heat_loss_0  0.658343  0.658343  1
   mass_balance  0.000000  0.000000  1
           pump  0.000000  0.000000  1


`demand_heat`(需要家の消費流量 `m_cons · CP · (t_node − T_RET)` = 流量×温度の双線形)が
相対違反で突出する。次いで `heat_loss` / `temp_mix` / `pressure_loss`(流量の2乗)と、
**物理の非線形項がそのまま緩和の弱点**として並ぶ。
`bottleneck_type` はこのランキング先頭を採用している。

### 3b. 双対境界の改善は何が押しているか(空間分枝の寄与)

`solve_and_attribute` は分枝ごとに双対境界の増分を記録し、分枝変数の種類
(0-1 / 整数 / spatial=連続への空間分枝)に帰属させる。
`gain_by_kind` でその内訳を出す。`spatial_share` はこの spatial の割合である。

In [7]:
from minlpkit.collectors.attribution import solve_and_attribute, gain_by_kind

d, summ = solve_and_attribute(dh.build_model(), time_limit=10, gap_limit=0.01)
gk = gain_by_kind(d)
print(gk.to_string(index=False))
total = gk["dual_gain"].sum()
spatial = gk.loc[gk["kind"] == "spatial", "dual_gain"].sum()
print(f"\n空間分枝の寄与(spatial_share)= {spatial/total:.1%}")

   kind   dual_gain
spatial 3100.824200
   root   81.103447

空間分枝の寄与(spatial_share)= 97.5%


## 4. 突き合わせ — 診断の結論

- 双対境界の改善の **大半(約97%)が空間分枝**由来。=SCIPは離散決定でなく
  **連続変数(流量・温度)への空間分枝で非凸緩和を締めることに労力を費やしている**。
- その空間分枝が集中する先が `demand_heat` などの**双線形物理制約**で、
  ルートLPでの相対違反も最大。

この2つが揃うので診断は `weak_relaxation`(重大)を出す。
つまり診断は「非線形制約の緩和が弱く、そこを空間分枝で埋めているのが律速」という
**物理的に納得できるボトルネック**を、可視化の素データから機械的に指摘している。

### 次の一手

recipe が示す直し方は「整数×連続の積 → `mk.linearize_product`、非線形1変数 → `mk.pwl_sos2`」。
このモデルの双線形は *連続×連続* なので linearize_product は直接は効かない
(変数境界タイト化や区分線形化が候補)。
**整数×連続の積を持つモデルで recipe が定量的に効く様子**は、
続く [ハンズオン(2)改善編](hands_on_improvement.ipynb) で
`mk.compare_variants` を使って before/after を測る。